## Overview

This tutorial covers advanced profiling techniques in `fasterbench` for identifying performance bottlenecks:

- **LayerProfiler**: Unified class for profiling speed, memory, size, and compute per layer
- **profile_layers()**: Quick per-layer latency profiling
- **Sweep functions**: Find optimal batch size, thread count, and resolution

## LayerProfiler: Unified Per-Layer Analysis

The `LayerProfiler` class provides a unified interface for profiling multiple metrics per layer. This is the recommended approach when you need comprehensive layer-level analysis.

In [ ]:
import torch
import pandas as pd
from torchvision.models import resnet18
from fasterbench.profiling import LayerProfiler

model = resnet18()
dummy = torch.randn(1, 3, 224, 224)

# Create profiler
profiler = LayerProfiler(model, dummy)

# Profile multiple metrics at once
results = profiler.profile(["speed", "size", "memory"], device="cpu", warmup=3, steps=10)

### Available Metrics

| Metric | Columns Added | Description |
|--------|---------------|-------------|
| `speed` | `speed_ms`, `speed_percent` | Forward pass latency per layer |
| `memory` | `memory_mib`, `memory_percent` | Output tensor size (activation memory) |
| `size` | `params`, `params_percent` | Parameter count per layer |
| `compute` | `macs`, `macs_percent` | MACs per layer (requires torchprofile) |

### Utility Methods: top() and summary()

The `LayerProfiler` provides convenient methods to quickly identify bottlenecks after profiling:

In [ ]:
# Get top 5 slowest layers
print("Top 5 slowest layers:")
for r in profiler.top("speed", n=5):
    print(f"  {r['name']:30} {r['speed_ms']:.3f} ms ({r['speed_percent']:.1f}%)")

# Get top 5 fastest layers (ascending order)
print("\nTop 5 fastest layers:")
for r in profiler.top("speed", n=5, ascending=True):
    print(f"  {r['name']:30} {r['speed_ms']:.3f} ms")

# Get layers with most parameters
print("\nTop 5 layers by parameter count:")
for r in profiler.top("size", n=5):
    print(f"  {r['name']:30} {r['params']:>12,} params")

Top 5 slowest layers:
  layer4.0.conv2                 0.478 ms (6.7%)
  layer4.1.conv2                 0.473 ms (6.6%)
  layer4.1.conv1                 0.471 ms (6.6%)
  maxpool                        0.410 ms (5.7%)
  layer3.1.conv1                 0.336 ms (4.7%)

Top 5 fastest layers:
  layer4.0.relu                  0.009 ms
  layer4.1.relu                  0.009 ms
  layer3.0.relu                  0.010 ms
  layer3.1.relu                  0.010 ms
  layer2.1.relu                  0.011 ms

Top 5 layers by parameter count:
  layer4.0.conv2                    2,359,296 params
  layer4.1.conv2                    2,359,296 params
  layer4.1.conv1                    2,359,296 params
  layer4.0.conv1                    1,179,648 params
  layer3.1.conv1                      589,824 params


In [ ]:
# Print formatted summary of all profiled metrics
profiler.summary(top=10)

═══ Speed (slowest) ═══════════════════════════════════
  layer4.0.conv2                           Conv2d             0.478 ms (  6.7%)
  layer4.1.conv2                           Conv2d             0.473 ms (  6.6%)
  layer4.1.conv1                           Conv2d             0.471 ms (  6.6%)
  maxpool                                  MaxPool2d          0.410 ms (  5.7%)
  layer3.1.conv1                           Conv2d             0.336 ms (  4.7%)
  layer3.0.conv2                           Conv2d             0.335 ms (  4.7%)
  layer3.1.conv2                           Conv2d             0.331 ms (  4.6%)
  conv1                                    Conv2d             0.325 ms (  4.5%)
  layer1.0.conv1                           Conv2d             0.317 ms (  4.4%)
  layer1.1.conv1                           Conv2d             0.309 ms (  4.3%)

═══ Parameters (largest) ══════════════════════════════
  layer4.0.conv2                           Conv2d             2,359,296 ( 20.2%)
  laye

## Quick Latency Profiling with profile_layers()

For quick per-layer latency analysis, use `profile_layers()` directly:

In [ ]:
from fasterbench import profile_layers

layers = profile_layers(model, dummy, device="cpu", warmup=5, steps=20)

print("Top 10 slowest layers:")
print("-" * 65)
for layer in layers[:10]:
    print(f"{layer['name']:40} {layer['type']:15} {layer['time_ms']:6.3f} ms ({layer['percent']:5.1f}%)")

Top 10 slowest layers:
-----------------------------------------------------------------
maxpool                                  MaxPool2d        0.509 ms (  6.9%)
layer4.0.conv2                           Conv2d           0.473 ms (  6.4%)
layer4.1.conv1                           Conv2d           0.450 ms (  6.1%)
layer4.1.conv2                           Conv2d           0.442 ms (  6.0%)
layer3.1.conv1                           Conv2d           0.367 ms (  5.0%)
layer3.0.conv2                           Conv2d           0.366 ms (  5.0%)
layer3.1.conv2                           Conv2d           0.363 ms (  4.9%)
conv1                                    Conv2d           0.354 ms (  4.8%)
layer1.0.conv1                           Conv2d           0.344 ms (  4.7%)
layer2.0.conv2                           Conv2d           0.344 ms (  4.7%)


In [ ]:
# Analyze by layer type
df = pd.DataFrame(layers)
print(f"\nTotal layers profiled: {len(df)}")
print(f"\nTime breakdown by layer type:")
print(df.groupby('type')['percent'].sum().sort_values(ascending=False))


Total layers profiled: 52

Time breakdown by layer type:
type
Conv2d               85.215934
MaxPool2d             6.914998
BatchNorm2d           5.217719
ReLU                  1.271526
AdaptiveAvgPool2d     0.695003
Linear                0.684820
Name: percent, dtype: float64


## Batch Size Sweeping

Find the optimal batch size for maximum throughput. Larger batches improve GPU utilization but eventually hit memory limits.

In [ ]:
from fasterbench import sweep_batch_sizes

if torch.cuda.is_available():
    results = sweep_batch_sizes(
        model,
        input_shape=(3, 224, 224),  # Shape WITHOUT batch dimension
        batch_sizes=[1, 2, 4, 8, 16, 32],
        device="cuda",
        warmup=10,
        steps=50
    )
    
    print("Batch Size Analysis:")
    print("-" * 60)
    print(f"{'Batch':>6} {'Latency':>10} {'Per-Sample':>12} {'Throughput':>12}")
    print(f"{'Size':>6} {'(ms)':>10} {'(ms)':>12} {'(inf/s)':>12}")
    print("-" * 60)
    for r in results:
        if 'throughput_s' in r and not pd.isna(r.get('mean_ms')):
            print(f"{r['batch_size']:>6} {r['mean_ms']:>10.2f} {r['latency_per_sample_ms']:>12.3f} {r['throughput_s']:>12.1f}")

Batch Size Analysis:
------------------------------------------------------------
 Batch    Latency   Per-Sample   Throughput
  Size       (ms)         (ms)      (inf/s)
------------------------------------------------------------
     1       0.72        0.725       1379.9
     2       0.77        0.386       2589.9
     4       1.00        0.250       3995.5
     8       1.18        0.148       6759.9
    16       1.65        0.103       9671.6
    32       2.87        0.090      11141.0


## Thread Count Sweeping (CPU)

For CPU inference, the number of threads significantly impacts performance. More threads isn't always better.

In [ ]:
from fasterbench import sweep_threads
import os

num_cores = os.cpu_count()
thread_counts = [t for t in [1, 2, 4, 8, 16, 32] if t <= num_cores]

results = sweep_threads(model, dummy, thread_counts=thread_counts, warmup=10, steps=30)

print("Thread Count Analysis:")
print("-" * 50)
print(f"{'Threads':>8} {'Latency (ms)':>15} {'Throughput':>15}")
print("-" * 50)
for r in results:
    print(f"{r['threads']:>8} {r['mean_ms']:>15.2f} {r['throughput_s']:>15.1f}")

Thread Count Analysis:
--------------------------------------------------
 Threads    Latency (ms)      Throughput
--------------------------------------------------
       1           28.56            35.0
       2           28.61            34.9
       4           28.52            35.1
       8           28.58            35.0
      16           28.55            35.0
      32           28.57            35.0


## Input Resolution Sweeping

For vision models, latency scales with input resolution:

In [ ]:
from fasterbench import sweep_latency

shapes = [
    (1, 3, 128, 128),
    (1, 3, 224, 224),
    (1, 3, 384, 384),
    (1, 3, 512, 512),
]

results = sweep_latency(model, shapes, device="cpu", warmup=5, steps=20)

print("Resolution Analysis:")
print("-" * 50)
print(f"{'Shape':>20} {'Latency (ms)':>15} {'Throughput':>12}")
print("-" * 50)
for r in results:
    print(f"{r['shape']:>20} {r['mean_ms']:>15.2f} {r['throughput_s']:>12.1f}")

Resolution Analysis:
--------------------------------------------------
               Shape    Latency (ms)   Throughput
--------------------------------------------------
         1×3×128×128           10.36         96.6
         1×3×224×224           28.64         34.9
         1×3×384×384         4211.25          0.2
         1×3×512×512        12085.58          0.1


## Summary

| Tool | Use Case |
|------|----------|
| `LayerProfiler` | Comprehensive per-layer analysis (speed, memory, size, compute) |
| `LayerProfiler.top()` | Get top N layers for a specific metric (with ascending option) |
| `LayerProfiler.summary()` | Print formatted summary of all profiled metrics |
| `profile_layers()` | Quick per-layer latency check |
| `sweep_batch_sizes()` | Find optimal batch size for throughput |
| `sweep_threads()` | Find optimal CPU thread count |
| `sweep_latency()` | Analyze latency vs input resolution |